# OWS Short Screen — Python vs. Excel Validation

This notebook runs the full pipeline against the reference Excel file and compares
all 24 factor scores row-by-row between Python output and Excel output.

**Acceptance criteria**: 95%+ of stocks match within ±0.001 per factor.

In [ ]:
import os
import sys
import shutil

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Add project root to path
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, project_root)

from src.ingest import read_upload, validate_columns, clean_dataframe
from src.transform import run_transforms
from src.score import load_config, run_scoring

## 1. Run the Python Pipeline

In [ ]:
# Read the reference Excel file (Data sheet)
excel_path = os.path.join(project_root, 'OWS Short Screen (March 2026).xlsx')
print(f'Reading: {excel_path}')

raw_df = read_upload(excel_path)
validate_columns(raw_df)
cleaned = clean_dataframe(raw_df)
print(f'Ingested: {len(cleaned)} rows, {len(cleaned.columns)} columns')

# Transform
transformed = run_transforms(cleaned.copy())
print(f'Transformed: {len(transformed)} rows')

# Score
config = load_config(os.path.join(project_root, 'config.yaml'))
scored = run_scoring(transformed.copy(), config)
print(f'Scored: {len(scored)} rows')
print(f'Overall score range: {scored["overall_score"].min():.3f} to {scored["overall_score"].max():.3f}')

## 2. Read Excel Screen Sheet (Reference Values)

In [ ]:
# Read the Screen sheet with computed values
screen_df = pd.read_excel(excel_path, sheet_name='Screen', header=2)
print(f'Screen sheet: {len(screen_df)} rows, {len(screen_df.columns)} columns')
print(f'Columns: {list(screen_df.columns)}')

In [ ]:
# Map Excel Screen column names to our Python factor names
EXCEL_TO_PYTHON_FACTORS = {
    'Abs. P/S Factor': 'abs_ps_factor',
    'Rel. P/S Factor': 'rel_ps_factor',
    'Abs. FCF% Factor': 'abs_fcf_factor',
    'Rel. FCF% Factor': 'rel_fcf_factor',
    'Decel Factor': 'decel_factor',
    'Accel Factor': 'accel_factor',
    'GM% Factor': 'gm_factor',
    'EBIT% Factor': 'ebit_factor',
    'Debt/EBITDA Factor': 'debt_ebitda_factor',
    'Debt/Sales Factor': 'debt_sales_factor',
    'Debt/EV Factor': 'debt_ev_factor',
    'Refinancing Risk Factor': 'refi_risk_factor',
    'Liquidity Risk Factor': 'liquidity_risk_factor',
    'FCF Conv. Factor': 'fcf_conv_factor',
    'Accrual Factor': 'accrual_factor',
    'DSO Factor': 'dso_factor',
    'DIO Factor': 'dio_factor',
    'DPO Factor': 'dpo_factor',
    'Def. Rev. Factor': 'def_rev_factor',
    'Dilution Factor': 'dilution_factor',
    'EBIT Adj. Factor': 'ebit_adj_factor',
    'EPS Adj. Factor': 'eps_adj_factor',
    'Short Int. Factor': 'short_int_factor',
    'Ratings Factor': 'ratings_factor',
}

print(f'Mapping {len(EXCEL_TO_PYTHON_FACTORS)} factor columns')

In [ ]:
# Join Python and Excel results on Ticker
# Excel Screen uses 'Ticker' column, Python uses 'ticker'
excel_tickers = screen_df['Ticker'].dropna().str.strip()
python_tickers = scored['ticker'].str.strip()

# Create join key
screen_df = screen_df.copy()
screen_df['join_ticker'] = excel_tickers
scored['join_ticker'] = python_tickers

common_tickers = set(screen_df['join_ticker'].dropna()) & set(scored['join_ticker'].dropna())
print(f'Common tickers: {len(common_tickers)}')
print(f'Excel-only: {len(set(screen_df["join_ticker"].dropna()) - common_tickers)}')
print(f'Python-only: {len(set(scored["join_ticker"].dropna()) - common_tickers)}')

## 3. Row-by-Row Factor Comparison

In [ ]:
# Build comparison for each factor
results = []

for excel_col, python_col in EXCEL_TO_PYTHON_FACTORS.items():
    if excel_col not in screen_df.columns:
        print(f'WARNING: Excel column "{excel_col}" not found in Screen sheet')
        continue
    if python_col not in scored.columns:
        print(f'WARNING: Python column "{python_col}" not found in scored data')
        continue

    # Merge on ticker
    excel_vals = screen_df[['join_ticker', excel_col]].dropna(subset=['join_ticker'])
    python_vals = scored[['join_ticker', python_col]].dropna(subset=['join_ticker'])

    merged = excel_vals.merge(python_vals, on='join_ticker', how='inner')

    # Coerce to numeric
    excel_numeric = pd.to_numeric(merged[excel_col], errors='coerce')
    python_numeric = pd.to_numeric(merged[python_col], errors='coerce')

    # Both must be non-NaN for comparison
    both_valid = excel_numeric.notna() & python_numeric.notna()
    n_valid = both_valid.sum()

    if n_valid == 0:
        results.append({
            'factor': python_col,
            'n_compared': 0,
            'pct_match_001': 0,
            'max_abs_dev': np.nan,
            'mean_abs_dev': np.nan,
            'status': 'NO DATA',
        })
        continue

    abs_diff = (excel_numeric[both_valid] - python_numeric[both_valid]).abs()
    match_001 = (abs_diff <= 0.001).sum()
    pct_match = match_001 / n_valid * 100

    results.append({
        'factor': python_col,
        'n_compared': int(n_valid),
        'pct_match_001': round(pct_match, 2),
        'max_abs_dev': round(abs_diff.max(), 6),
        'mean_abs_dev': round(abs_diff.mean(), 6),
        'status': 'PASS' if pct_match >= 95 else 'FAIL',
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 4. Overall Score Comparison

In [ ]:
# Compare overall scores
if 'Overall Score' in screen_df.columns:
    excel_overall = screen_df[['join_ticker', 'Overall Score']].dropna(subset=['join_ticker'])
    python_overall = scored[['join_ticker', 'overall_score']].dropna(subset=['join_ticker'])

    merged_overall = excel_overall.merge(python_overall, on='join_ticker', how='inner')
    excel_os = pd.to_numeric(merged_overall['Overall Score'], errors='coerce')
    python_os = pd.to_numeric(merged_overall['overall_score'], errors='coerce')

    both_valid = excel_os.notna() & python_os.notna()
    abs_diff = (excel_os[both_valid] - python_os[both_valid]).abs()

    print(f'Overall Score comparison ({both_valid.sum()} stocks):')
    print(f'  Match within ±0.001: {(abs_diff <= 0.001).sum()} ({(abs_diff <= 0.001).mean()*100:.1f}%)')
    print(f'  Match within ±0.01:  {(abs_diff <= 0.01).sum()} ({(abs_diff <= 0.01).mean()*100:.1f}%)')
    print(f'  Match within ±0.05:  {(abs_diff <= 0.05).sum()} ({(abs_diff <= 0.05).mean()*100:.1f}%)')
    print(f'  Max absolute deviation: {abs_diff.max():.6f}')
    print(f'  Mean absolute deviation: {abs_diff.mean():.6f}')
else:
    print('Overall Score column not found in Screen sheet')

## 5. M-Score Comparison

In [ ]:
# Compare M-Scores
if 'M-Score' in screen_df.columns:
    excel_ms = screen_df[['join_ticker', 'M-Score']].dropna(subset=['join_ticker'])
    python_ms = scored[['join_ticker', 'mscore']].dropna(subset=['join_ticker'])

    merged_ms = excel_ms.merge(python_ms, on='join_ticker', how='inner')
    excel_mscore = pd.to_numeric(merged_ms['M-Score'], errors='coerce')
    python_mscore = pd.to_numeric(merged_ms['mscore'], errors='coerce')

    both_valid = excel_mscore.notna() & python_mscore.notna()
    abs_diff = (excel_mscore[both_valid] - python_mscore[both_valid]).abs()

    print(f'M-Score comparison ({both_valid.sum()} stocks):')
    print(f'  Match within ±0.001: {(abs_diff <= 0.001).sum()} ({(abs_diff <= 0.001).mean()*100:.1f}%)')
    print(f'  Max absolute deviation: {abs_diff.max():.6f}')
    print(f'  Mean absolute deviation: {abs_diff.mean():.6f}')
else:
    print('M-Score column not found in Screen sheet')

## 6. Summary

In [ ]:
# Summary
passing = results_df[results_df['status'] == 'PASS']
failing = results_df[results_df['status'] == 'FAIL']
no_data = results_df[results_df['status'] == 'NO DATA']

print(f'\n=== VALIDATION SUMMARY ===')
print(f'Factors passing (>=95% match within ±0.001): {len(passing)}/{len(results_df)}')
print(f'Factors failing: {len(failing)}/{len(results_df)}')
print(f'Factors with no data: {len(no_data)}/{len(results_df)}')

if len(failing) > 0:
    print(f'\nFAILING FACTORS (these are bugs):')
    print(failing[['factor', 'pct_match_001', 'max_abs_dev', 'mean_abs_dev']].to_string(index=False))